# Session 2: one storey, shaken at the base
**A. Real-world problem.** A one-storey building has a roof mass m on columns of total
sideways stiffness k. The ground moves with acceleration a_g(t). Typical dominant ground
shaking in a soft-soil city is around 1 s per cycle. How big does the sway get, and how stiff
must the columns be so that the building is not at the mercy of that rhythm?

**Plan.** (1) equation of motion and natural period, (2) the amplification factor,
(3) check it against a step-by-step numerical solution, (4) solve a design problem.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
ZETA = 0.05          # typical damping ratio of a building (about 5 percent of critical)

## Step 1: equation of motion
Let u be the roof displacement *relative to the ground*. Newton's law on the roof gives

    m u'' + c u' + k u = -m a_g(t)

Divide by m, with w = sqrt(k/m) and damping ratio zeta = c / (2 m w):

    u'' + 2 zeta w u' + w^2 u = -a_g(t)

Period T = 2 pi / w. For N storeys of a typical frame building a rough *empirical* guide is
T about 0.1 N seconds (not a law).

In [ ]:
def omega(k, m):
    """Natural angular frequency w = sqrt(k / m) in rad/s."""
    return float(np.sqrt(k / m))


def period(k, m):
    """Natural period T = 2 pi / w in seconds."""
    return float(2 * np.pi / omega(k, m))


def rule_of_thumb_period(storeys):
    """T = 0.1 N seconds: rough empirical guide for typical frame buildings."""
    return 0.1 * storeys

## Step 2: harmonic ground motion and the amplification factor
Let the ground accelerate as a_g = a0 sin(W t) with forcing frequency W, and let r = W / w be
the *frequency ratio*. After the start-up transient dies, the building sways with amplitude

    u_max = (a0 / w^2) D(r),     D(r) = 1 / sqrt((1 - r^2)^2 + (2 zeta r)^2)

a0 / w^2 is the sway if the acceleration were applied as a constant push. D is the
*amplification factor*. Its maximum is near r = 1 and equals about 1 / (2 zeta).

In [ ]:
def amplification(r, zeta=ZETA):
    """D(r), the dynamic amplification factor (no unit)."""
    r = np.asarray(r, float)
    return 1.0 / np.sqrt((1 - r ** 2) ** 2 + (2 * zeta * r) ** 2)


def peak_amplification(zeta=ZETA):
    """Exact maximum of D (at r = sqrt(1 - 2 zeta^2), needs zeta < 0.7071) and where it occurs."""
    r = np.sqrt(1 - 2 * zeta ** 2)
    return float(amplification(r, zeta)), float(r)


def sway_amplitude(a0, k, m, t_ground, zeta=ZETA):
    """Steady sway amplitude (m) of the roof relative to the ground for ground acceleration amplitude a0 (m/s^2)."""
    w = omega(k, m)
    r = (2 * np.pi / t_ground) / w
    return float(a0 / w ** 2 * amplification(r, zeta))

## Step 3: check it by brute force
Step u'' + 2 zeta w u' + w^2 u = -a0 sin(W t) forward with the 4th-order Runge-Kutta
method until the transient has died, then read off the amplitude. It should match the formula.

In [ ]:
def sway_amplitude_numeric(a0, k, m, t_ground, zeta=ZETA, steps_per_period=200):
    w = omega(k, m)
    W = 2 * np.pi / t_ground
    h = min(t_ground, 2 * np.pi / w) / steps_per_period
    t_end = 12.0 / (zeta * w) + 20 * t_ground
    n = int(t_end / h)

    def f(t, y):
        return np.array([y[1], -a0 * np.sin(W * t) - 2 * zeta * w * y[1] - w * w * y[0]])

    y = np.zeros(2)
    t = 0.0
    peak = 0.0
    last = n - int(3 * max(t_ground, 2 * np.pi / w) / h)
    for i in range(n):
        k1 = f(t, y); k2 = f(t + h / 2, y + h / 2 * k1); k3 = f(t + h / 2, y + h / 2 * k2); k4 = f(t + h, y + h * k3)
        y = y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t += h
        if i >= last:
            peak = max(peak, abs(y[0]))
    return float(peak)

## Step 4: a design problem
The ground has a dominant period T_g = 1 s. We want the amplification factor to stay at or
below D_max = 2. There are two ways to get there: make the building *stiffer* (r below 1,
short period) or *much more flexible* (r above 1, long period, as with base isolation).
Solve D(r) = D_max for r by bisection on each side of the resonance, then turn r into a
stiffness with  T = r T_g  and  k = m (2 pi / T)^2.

In [ ]:
def stiffness_for_period(m, t):
    """k (N/m) that gives period t (s) for mass m (kg)."""
    return float(m * (2 * np.pi / t) ** 2)


def r_for_amplification(dmax, zeta=ZETA, side="stiff"):
    """The frequency ratio r where D(r) = dmax: below the resonance ('stiff') or above it ('flexible')."""
    rp = peak_amplification(zeta)[1]
    if dmax >= peak_amplification(zeta)[0]:
        raise ValueError("dmax must be below the resonance peak")
    lo, hi = (0.0, rp) if side == "stiff" else (rp, 20.0)
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        above = amplification(mid, zeta) > dmax
        if side == "stiff":
            lo, hi = (mid, hi) if not above else (lo, mid)
        else:
            lo, hi = (lo, mid) if not above else (mid, hi)
    return 0.5 * (lo + hi)


def design_stiffness(m, t_ground, dmax, zeta=ZETA, side="stiff"):
    """Stiffness (N/m) keeping the amplification factor at dmax for a ground period t_ground.
    r = (W / w) = T / t_ground, so the building period is T = r t_ground."""
    r = r_for_amplification(dmax, zeta, side)
    return stiffness_for_period(m, r * t_ground)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m = 2.0e5
    k0 = stiffness_for_period(m, 1.0)
    print(f"m = {m:.0f} kg, k0 = {k0:.4g} N/m gives T = {period(k0, m):.4f} s (resonant with a 1 s ground)")
    print("peak amplification at 5%:", peak_amplification(0.05), " 1/(2 zeta) =", 1 / 0.1)
    ks = design_stiffness(m, 1.0, 2.0)
    kf = design_stiffness(m, 1.0, 2.0, side="flexible")
    print(f"stiffer design: k = {ks:.4g} N/m (x{ks / k0:.3f}), T = {period(ks, m):.4f} s")
    print(f"flexible design: k = {kf:.4g} N/m (x{kf / k0:.3f}), T = {period(kf, m):.4f} s")
    r = np.linspace(0, 3, 601)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for z in (0.02, 0.05, 0.10, 0.20):
        ax[0].plot(r, amplification(r, z), label=f"zeta = {z}")
    ax[0].axhline(2, color="red", ls="--", lw=0.8)
    ax[0].set_ylim(0, 12)
    ax[0].set_xlabel("frequency ratio r = W / w (no unit)")
    ax[0].set_ylabel("amplification factor D (no unit)")
    ax[0].set_title("Resonance curves")
    ax[0].grid(alpha=0.3)
    ax[0].legend(fontsize=8)
    # numerical dots on the formula
    for rr in (0.5, 1.0, 1.5):
        tg = 1.0
        w_target = 2 * np.pi / tg / rr
        kk = m * w_target ** 2
        num = sway_amplitude_numeric(1.0, kk, m, tg)
        ax[0].plot(rr, num / (1.0 / w_target ** 2), "ko", ms=5)
        print(f"r = {rr}: numeric D = {num * w_target ** 2:.4f}, formula {amplification(rr):.4f}")
    kk = np.linspace(2e6, 4e7, 200)
    ax[1].plot(kk / 1e6, [sway_amplitude(2.0, x, m, 1.0) * 1000 for x in kk])
    ax[1].axvline(k0 / 1e6, color="gray", ls=":", label="k giving T = 1 s")
    ax[1].axvline(ks / 1e6, color="green", ls="--", label="stiffer design (D = 2)")
    ax[1].axvline(kf / 1e6, color="purple", ls="--", label="flexible design (D = 2)")
    ax[1].set_xlabel("storey stiffness k (MN/m)")
    ax[1].set_ylabel("steady sway amplitude (mm)")
    ax[1].set_title("Ground: a0 = 2 m/s^2, period 1 s")
    ax[1].set_yscale("log")
    ax[1].grid(alpha=0.3, which="both")
    ax[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUT / "resonance_and_design.png", dpi=120)

## Where this stops
One lump cannot show how floors bend over each other; Session 3 gives each floor its own mass.
Steady harmonic shaking is also kinder than a real record, which has many frequencies at once.

## Coding tasks
1. **Easy.** Run `sway_amplitude(2.0, k, m, 1.0)` for k = 1.563e7 and compare with the answer in Section E.
2. **Stretch.** Use `sway_amplitude_numeric` and `amplification` to make a table of D at r = 0.5, 1 and 1.5 and show that the two agree within 0.5 percent.
3. **Explain.** Explain why the flexible design in the worked example sways further than the stiff design even though both have D = 2.

## Assessment
- **A. Conceptual.** Why does the amplification factor go above 1 near r = 1 but fall below 1 for large r? What does damping change?
- **B. Numerical.** A storey has m = 100 000 kg and k = 4e6 N/m, damping 5 percent. The ground has period 1 s and acceleration amplitude 2 m/s^2. Find w, T, r, D and the steady sway amplitude.
- **C. Graph and simulation interpretation.** On the resonance curves, read D at r = 1 for zeta = 0.02, 0.05, 0.1 and 0.2, and D at r = 1.5 for zeta = 0.05. What does each tell you?
- **D. Design and optimisation challenge.** The ground period is 1.5 s (softer soil) and the roof mass is 200 000 kg. Find the stiffness and the building period for D = 2 on the stiff side and on the flexible side, with the sway amplitude for a0 = 2 m/s^2. Which design would you prefer and why?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    m = 2.0e5
    k0 = L.stiffness_for_period(m, 1.0)
    checks.append((f"worked example: m = 200000 kg and T = 1 s need k = 7.896e6 N/m (got {k0:.4g}); w = 6.283 rad/s",
                   abs(k0 - 7.896e6) < 5e3 and abs(L.omega(k0, m) - 2 * np.pi) < 1e-9))
    checks.append(("period(k0, m) returns 1 s", abs(L.period(k0, m) - 1.0) < 1e-12))
    checks.append(("rule of thumb: 10 storeys is 1.0 s, 5 storeys 0.5 s",
                   abs(L.rule_of_thumb_period(10) - 1.0) < 1e-12 and abs(L.rule_of_thumb_period(5) - 0.5) < 1e-12))

    # amplification by hand
    d1 = L.amplification(1.0, 0.05)
    checks.append(("D(1) = 1 / (2 zeta) = 10 for zeta = 0.05", abs(d1 - 10.0) < 1e-9))
    checks.append(("D(0) = 1 (a very slow push moves the building statically)", abs(L.amplification(0.0) - 1.0) < 1e-12))
    checks.append(("hand calculation: D(0.5) = 1 / sqrt(0.75^2 + 0.05^2) = 1.3304",
                   abs(L.amplification(0.5) - 1 / np.sqrt(0.75 ** 2 + 0.05 ** 2)) < 1e-12 and abs(L.amplification(0.5) - 1.3304) < 5e-4))
    checks.append(("D(2) = 0.3326: a fast shake barely moves the building", abs(L.amplification(2.0) - 0.3326) < 5e-4))
    pk, rp = L.peak_amplification(0.05)
    grid = np.linspace(0, 3, 300001)
    checks.append((f"exact peak 10.0125 at r = 0.9975 matches a fine grid search (got {pk:.4f}, {rp:.4f})",
                   abs(pk - 10.0125) < 5e-4 and abs(rp - 0.9975) < 5e-4 and abs(L.amplification(grid, 0.05).max() - pk) < 1e-5))
    checks.append(("peak is close to 1/(2 zeta) for small damping: within 0.2 percent at zeta = 0.05",
                   abs(pk - 10.0) / 10.0 < 2e-3))
    checks.append(("more damping, lower peak: zeta 0.02 gives 25.0, zeta 0.2 gives 2.5",
                   abs(L.amplification(1.0, 0.02) - 25.0) < 1e-9 and abs(L.amplification(1.0, 0.2) - 2.5) < 1e-9))

    # numerical solution agrees with the formula
    ok = True
    for rr in (0.5, 1.0, 1.5):
        w = 2 * np.pi / rr
        kk = m * w ** 2
        num = L.sway_amplitude_numeric(1.0, kk, m, 1.0) * w ** 2
        ok &= abs(num - L.amplification(rr)) / L.amplification(rr) < 5e-3
    checks.append(("RK4 steady amplitude matches D(r) within 0.5 percent at r = 0.5, 1.0, 1.5", ok))
    checks.append(("sway_amplitude uses a0 / w^2 * D: 2 m/s^2 on k0 gives 0.5066 m",
                   abs(L.sway_amplitude(2.0, k0, m, 1.0) - 2.0 / (2 * np.pi) ** 2 * 10.0) < 1e-9
                   and abs(L.sway_amplitude(2.0, k0, m, 1.0) - 0.5066) < 5e-4))

    # design problem
    rs = L.r_for_amplification(2.0, 0.05, "stiff")
    rf = L.r_for_amplification(2.0, 0.05, "flexible")
    checks.append((f"D = 2 at r = 0.7107 (stiff side) and r = 1.2186 (flexible side) (got {rs:.4f}, {rf:.4f})",
                   abs(rs - 0.7107) < 5e-4 and abs(rf - 1.2186) < 5e-4
                   and abs(L.amplification(rs) - 2) < 1e-9 and abs(L.amplification(rf) - 2) < 1e-9))
    ks = L.design_stiffness(m, 1.0, 2.0)
    kf = L.design_stiffness(m, 1.0, 2.0, side="flexible")
    checks.append((f"design: stiffer building needs k = 1.563e7 N/m, 1.98 times k0 (got {ks:.4g}, x{ks / k0:.3f})",
                   abs(ks - 1.563e7) < 2e4 and abs(ks / k0 - 1.980) < 2e-3 and abs(L.period(ks, m) - 0.7107) < 5e-4))
    checks.append((f"design: flexible building needs k = 5.317e6 N/m, 0.673 times k0 (got {kf:.4g}, x{kf / k0:.3f})",
                   abs(kf - 5.317e6) < 2e4 and abs(kf / k0 - 0.673) < 2e-3 and abs(L.period(kf, m) - 1.2186) < 5e-4))
    checks.append(("design stiffness really gives D = 2 when fed back through the period",
                   abs(L.amplification(L.period(ks, m) / 1.0) - 2) < 1e-9))
    # edge cases
    try:
        L.r_for_amplification(20.0, 0.05)
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge: asking for D above the resonance peak is refused", raised))
    checks.append(("edge: the peak formula needs zeta below 0.7071; at zeta = 0.7 the maximum is D = 1.000 near r = 0.14 (no real resonance)",
                   abs(L.peak_amplification(0.7)[0] - 1 / (2 * 0.7 * np.sqrt(1 - 0.49))) < 1e-9
                   and abs(L.peak_amplification(0.7)[0] - 1.0) < 5e-3 and abs(L.peak_amplification(0.7)[1] - 0.1414) < 5e-3))
    checks.append(("D falls below 1 for r above 1.41 (D = 1.000 at r = 1.4107 for zeta = 0.05)", abs(L.amplification(1.4107) - 1.0) < 2e-3 and L.amplification(1.5) < 1))
    # assessment answers quoted in Section F
    checks.append(("B: m = 100000 kg, k = 4e6 N/m, ground 1 s: w = 6.325 rad/s, T = 0.9935 s, D = 9.98, sway 0.499 m for a0 = 2 m/s^2",
                   abs(L.omega(4e6, 1e5) - 6.325) < 5e-4 and abs(L.period(4e6, 1e5) - 0.9935) < 5e-5
                   and abs(float(L.amplification(L.period(4e6, 1e5))) - 9.98) < 5e-3 and abs(L.sway_amplitude(2.0, 4e6, 1e5, 1.0) - 0.499) < 5e-4))
    checks.append(("C: at r = 1 the factor is 1/(2 zeta): 25, 10, 5, 2.5 for zeta = 0.02, 0.05, 0.1, 0.2; D(1.5) = 0.794",
                   all(abs(L.amplification(1.0, z) - 1 / (2 * z)) < 1e-9 for z in (0.02, 0.05, 0.1, 0.2)) and abs(L.amplification(1.5) - 0.794) < 5e-4))
    kst = L.design_stiffness(2e5, 1.5, 2.0)
    kfl = L.design_stiffness(2e5, 1.5, 2.0, side="flexible")
    checks.append((f"D: ground period 1.5 s, m = 200000 kg, D = 2: stiff k = 6.948e6 N/m (T = 1.066 s, sway 0.115 m), flexible k = 2.363e6 N/m (T = 1.828 s, sway 0.339 m)",
                   abs(kst - 6.948e6) < 2e3 and abs(L.period(kst, 2e5) - 1.066) < 5e-4 and abs(L.sway_amplitude(2.0, kst, 2e5, 1.5) - 0.115) < 5e-4
                   and abs(kfl - 2.363e6) < 2e3 and abs(L.period(kfl, 2e5) - 1.828) < 5e-4 and abs(L.sway_amplitude(2.0, kfl, 2e5, 1.5) - 0.339) < 5e-4))
    checks.append(("a ground period of 1.5 s with T = 1.5 s needs k = 3.509e6 N/m for m = 200000 kg", abs(L.stiffness_for_period(2e5, 1.5) - 3.509e6) < 2e3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
